# TRỰC QUAN HÓA DỮ LIỆU WORLD CUP (1930 - 2026)
## Phân tích Mục 1 - 3 (Câu hỏi 1) & Mục 4 - 5 (Câu hỏi 2)
Tài liệu quy chuẩn: khdl.txt
Dữ liệu sử dụng: production/worldcup_master_cleaned.csv & production/dim_tournaments.csv


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'Segoe UI', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', palette='deep')

# Load dataset
df = pd.read_csv('../production/worldcup_master_cleaned.csv')
dim_tourn = pd.read_csv('../production/dim_tournaments.csv')
print('Kích thước tập dữ liệu Master:', df.shape)
print('Số kỳ World Cup:', len(dim_tourn))


### 1. Quy mô giải đấu và lượng khán giả mỗi trận (Combo Chart)
* **Ý nghĩa:** Trả lời việc tăng số trận có đi kèm với việc sụt giảm khán giả trung bình mỗi trận hay không?


In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6), dpi=150)
years = dim_tourn['Year'].values
matches = dim_tourn['MatchesPlayed'].values
mean_att = dim_tourn['Mean_Attendance'].values

colors = ['#2b5c8f' if y < 2026 else '#e26d5c' for y in years]
bars = ax1.bar(years, matches, color=colors, alpha=0.75, width=2.4, edgecolor='black', linewidth=0.8, label='Tổng số trận (Cột trái)')
bars[-1].set_hatch('///')

ax2 = ax1.twinx()
hist_mask = years <= 2022
ax2.plot(years[hist_mask], mean_att[hist_mask], color='#1b4965', marker='o', linewidth=2.5, label='Khán giả TB/trận (Lịch sử)')
sim_mask = years >= 2022
ax2.plot(years[sim_mask], mean_att[sim_mask], color='#d90429', marker='s', linewidth=2.5, linestyle='--', label='Khán giả TB/trận (2026 Mô phỏng)')

for ey, elab in zip([1982, 1998, 2026], ['Mở rộng 24 đội', 'Mở rộng 32 đội', 'Mở rộng 48 đội']):
    ax1.axvline(x=ey - 2, color='#7f7f7f', linestyle=':', linewidth=1.5, alpha=0.8)
    ax1.text(ey - 2, 100, f'  {elab}', rotation=90, verticalalignment='top', fontsize=9, fontweight='semibold')

ax1.set_xlabel('Năm diễn ra World Cup', fontweight='bold')
ax1.set_ylabel('Tổng số trận đấu', fontweight='bold', color='#2b5c8f')
ax2.set_ylabel('Khán giả trung bình / trận (người)', fontweight='bold', color='#1b4965')
ax1.set_xticks(years)
ax1.set_xticklabels(years, rotation=45, ha='right')
ax1.set_ylim(0, 115)
ax2.set_ylim(15000, 80000)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))

lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, loc='upper left')
plt.title('BIỂU ĐỒ 1: QUY MÔ SỐ TRẬN VÀ LƯỢNG KHÁN GIẢ TRUNG BÌNH MỖI TRẬN (1930 - 2026)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


### 2. Sức hút khán giả theo từng vòng đấu (Line Chart)
* **Ý nghĩa:** Trả lời xu hướng tăng hoặc giảm khán giả tập trung chủ yếu ở giai đoạn/nhóm trận nào?


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
group_df = df[df['Stage_Detail'] == 'First Group Stage']
group_att = group_df.groupby('Year')['Attendance'].mean()

ko_df = df[df['Stage_Group'].isin(['2. Early Knockout', '3. Deep Knockout'])]
ko_att = ko_df.groupby('Year')['Attendance'].mean()
all_years = sorted(df['Year'].unique())

hist_years = [y for y in all_years if y <= 2022 and y in group_att.index]
ax.plot(hist_years, [group_att[y] for y in hist_years], color='#2a9d8f', marker='o', linewidth=2.5, label='Vòng bảng đầu tiên (Lịch sử)')
ax.plot([2022, 2026], [group_att[2022], group_att[2026]], color='#2a9d8f', marker='o', linewidth=2.5, linestyle='--', label='Vòng bảng (2026 Mô phỏng)')

hist_ko_years = [y for y in all_years if y <= 2022 and y in ko_att.index]
ax.plot(hist_ko_years, [ko_att[y] for y in hist_ko_years], color='#e76f51', marker='s', linewidth=2.5, label='Vòng Knockout (Lịch sử)')
ax.plot([2022, 2026], [ko_att[2022], ko_att[2026]], color='#e76f51', marker='s', linewidth=2.5, linestyle='--', label='Vòng Knockout (2026 Mô phỏng)')

for ey, elab in zip([1982, 1998, 2026], ['Mở rộng 24 đội', 'Mở rộng 32 đội', 'Mở rộng 48 đội']):
    ax.axvline(x=ey, color='#7f7f7f', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.text(ey - 0.4, 76000, f'{elab}', rotation=90, verticalalignment='top', fontsize=9, fontweight='semibold')

ax.set_xlabel('Năm diễn ra World Cup', fontweight='bold')
ax.set_ylabel('Khán giả trung bình / trận (người)', fontweight='bold')
ax.set_xticks(all_years)
ax.set_xticklabels(all_years, rotation=45, ha='right')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x):,}'))
ax.set_ylim(10000, 85000)
ax.legend(loc='upper left')
plt.title('BIỂU ĐỒ 2: SO SÁNH SỨC HÚT KHÁN GIẢ GIỮA VÒNG BẢNG VÀ VÒNG KNOCKOUT', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


### 3. Phân phối tỷ lệ lấp đầy sân vận động (Boxplot)
* **Ý nghĩa:** Khi đối chiếu với sức chứa thực tế của sân, mức độ lấp đầy khán đài có sự khác biệt rõ rệt giữa các giai đoạn không?


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
box_df = df.copy()
box_df['Stage_Type'] = box_df['Stage_Group'].apply(lambda x: 'Vòng bảng' if x == '1. Group Stage' else 'Vòng Knockout')
era_labels = {
    '16 Teams (1930-1978)': '16 Đội (1930-1978)',
    '24 Teams (1982-1994)': '24 Đội (1982-1994)',
    '32 Teams (1998-2022)': '32 Đội (1998-2022)',
    '48 Teams (2026)': '48 Đội (2026 - Mô phỏng)'
}
box_df['Era_Label'] = box_df['Era_Format'].map(era_labels)
era_order = list(era_labels.values())

palette = {'Vòng bảng': '#457b9d', 'Vòng Knockout': '#e63946'}
sns.boxplot(data=box_df, x='Era_Label', y='Occupancy_Rate', hue='Stage_Type', order=era_order, palette=palette, ax=ax, width=0.6)
ax.axhline(100, color='#2b2d42', linestyle='--', linewidth=1.2, alpha=0.7, label='Mức 100% sức chứa')
ax.set_xlabel('Kỷ nguyên thể thức', fontweight='bold')
ax.set_ylabel('Tỷ lệ lấp đầy sân (%) [Occupancy Rate]', fontweight='bold')
ax.set_ylim(20, 108)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x)}%'))
ax.legend(title='Phân loại vòng đấu', loc='lower right')
plt.title('BIỂU ĐỒ 3: PHÂN PHỐI TỶ LỆ LẤP ĐẦY SÂN VẬN ĐỘNG THEO 4 KỶ NGUYÊN', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


### 4. Cơ cấu cách biệt tỷ số ở vòng bảng (100% Stacked Bar)
* **Ý nghĩa:** Tỷ lệ các trận đấu sát nút có giảm đi, hoặc tỷ lệ các trận thắng đậm có gia tăng khi giải mở rộng không?


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6.5), dpi=150)
group_df = df[df['Stage_Detail'] == 'First Group Stage'].copy()
ct = pd.crosstab(group_df['Year'], group_df['Margin_Category'], normalize='index') * 100
cat_order = ['1. Sat Nut / Hoa (<=1 ban)', '2. Cach Biet (2 ban)', '3. Thang Dam (>=3 ban)']
ct = ct[cat_order]

years = ct.index.values
sat_nut = ct['1. Sat Nut / Hoa (<=1 ban)'].values
cach_biet = ct['2. Cach Biet (2 ban)'].values
thang_dam = ct['3. Thang Dam (>=3 ban)'].values

colors = ['#2a9d8f', '#e9c46a', '#e76f51']
bar_width = 2.4

p1 = ax.bar(years, sat_nut, width=bar_width, color=colors[0], label='Sát nút / Hòa (≤ 1 bàn)')
p2 = ax.bar(years, cach_biet, bottom=sat_nut, width=bar_width, color=colors[1], label='Cách biệt vừa (2 bàn)')
p3 = ax.bar(years, thang_dam, bottom=sat_nut + cach_biet, width=bar_width, color=colors[2], label='Thắng đậm áp đảo (≥ 3 bàn)')
p1[-1].set_hatch('///')
p2[-1].set_hatch('///')
p3[-1].set_hatch('///')

for ey, elab in zip([1982, 1998, 2026], ['Mở rộng 24 đội', 'Mở rộng 32 đội', 'Mở rộng 48 đội']):
    ax.axvline(x=ey - 2, color='#333333', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.text(ey - 2, 102, elab, ha='center', fontsize=9, fontweight='semibold')

ax.set_xlabel('Năm diễn ra World Cup (Có vòng bảng)', fontweight='bold')
ax.set_ylabel('Tỷ lệ phần trăm số trận đấu (%)', fontweight='bold')
ax.set_xticks(years)
ax.set_xticklabels(years, rotation=45, ha='right')
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x)}%'))
ax.legend(loc='lower left')
plt.title('BIỂU ĐỒ 4: CƠ CẤU CÁCH BIỆT TỶ SỐ TRONG CÁC TRẬN ĐẤU VÒNG BẢNG (1930 - 2026)', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()


### 5. Tỷ lệ thắng đậm ở vòng bảng so với vòng knockout đầu tiên (Dual Line)
* **Ý nghĩa:** Mức độ chênh lệch vượt trội tập trung chủ yếu ở vòng bảng hay lan sang cả vòng knockout đầu tiên?


In [ ]:
fig, ax = plt.subplots(figsize=(14, 6), dpi=150)
group_blowout = group_df.groupby('Year')['Margin_Category'].apply(lambda x: (x == '3. Thang Dam (>=3 ban)').mean() * 100)
early_ko = df[df['Stage_Detail'] == 'Early Knockout']
early_ko_blowout = early_ko.groupby('Year')['Margin_Category'].apply(lambda x: (x == '3. Thang Dam (>=3 ban)').mean() * 100)

all_years = sorted(list(set(group_blowout.index.tolist() + early_ko_blowout.index.tolist())))
grp_hist = [y for y in group_blowout.index if y <= 2022]
ax.plot(grp_hist, [group_blowout[y] for y in grp_hist], color='#1d3557', marker='o', linewidth=2.5, label='Vòng bảng đầu tiên (Lịch sử)')
ax.plot([2022, 2026], [group_blowout[2022], group_blowout[2026]], color='#1d3557', marker='o', linewidth=2.5, linestyle='--', label='Vòng bảng (2026 Mô phỏng)')

ko_hist = [y for y in early_ko_blowout.index if y <= 2022]
ax.plot(ko_hist, [early_ko_blowout[y] for y in ko_hist], color='#e63946', marker='s', linewidth=2.5, label='Vòng Knockout đầu tiên (Lịch sử)')
ax.plot([2022, 2026], [early_ko_blowout[2022], early_ko_blowout[2026]], color='#e63946', marker='s', linewidth=2.5, linestyle='--', label='Vòng Knockout đầu tiên (2026 Mô phỏng)')

for ey, elab in zip([1982, 1998, 2026], ['Mở rộng 24 đội', 'Mở rộng 32 đội', 'Mở rộng 48 đội']):
    ax.axvline(x=ey, color='#6c757d', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.text(ey - 0.4, 52, elab, rotation=90, verticalalignment='top', fontsize=9, fontweight='semibold')

ax.axvspan(1954, 1982, color='#ced4da', alpha=0.25, label='1954-1982: Không có vòng R16')
ax.set_xlabel('Năm diễn ra World Cup', fontweight='bold')
ax.set_ylabel('Tỷ lệ trận thắng đậm (≥ 3 bàn) [%]', fontweight='bold')
ax.set_xticks(all_years)
ax.set_xticklabels(all_years, rotation=45, ha='right')
ax.set_ylim(-2, 60)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{int(x)}%'))
ax.legend(loc='upper right')
plt.title('BIỂU ĐỒ 5: TỶ LỆ TRẬN THẮNG ĐẬM Ở VÒNG BẢNG SO VỚI VÒNG KNOCKOUT ĐẦU TIÊN', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
